# IndPenSim Dataset — Overview & EDA

**목적**: 100배치 페니실린 발효 공정 시뮬레이션 데이터에 대한 탐색적 분석  
**데이터**: `100_Batches_IndPenSim_V3.csv` (시계열), `100_Batches_IndPenSim_Statistics.csv` (배치 요약)  

---
**분석 순서**
1. 라이브러리 및 데이터 로드
2. 데이터 기본 구조 확인
3. 배치 통계 분석 (Fault 비율, 수율 분포)
4. 공정 변수 시계열 탐색
5. 라만 분광 데이터 탐색
6. 결측치 분석
7. Fault vs Normal 배치 비교
8. 요약 및 연구 방향

## 1. 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

DATA_DIR   = Path('../data/raw')
STATS_PATH = DATA_DIR / '100_Batches_IndPenSim_Statistics.csv'
MAIN_PATH  = DATA_DIR / '100_Batches_IndPenSim_V3.csv'

BATCH_COL       = 'Batch reference(Batch_ref:Batch ref)'
BATCH_COL_STATS = 'Batch ref'
FAULT_COL       = 'Fault ref(0-NoFault 1-Fault)'
PAT_COL         = '2-PAT control(PAT_ref:PAT ref)'
TIME_COL        = 'Time (h)'
YIELD_COL       = 'Penicllin_yield_total (kg)'

In [ ]:
df_stats = pd.read_csv(STATS_PATH)
df_stats.columns = df_stats.columns.str.strip()
print('Statistics shape:', df_stats.shape)
df_stats.head()

In [ ]:
# 대용량 파일 — 수십 초 소요
df = pd.read_csv(MAIN_PATH)
df.columns = df.columns.str.strip()
print('Main data shape:', df.shape)
df.head(3)

## 2. 데이터 기본 구조 확인

In [ ]:
raman_cols   = [c for c in df.columns if c.strip().lstrip('-').isdigit()]
process_cols = [c for c in df.columns if c not in raman_cols]

print(f'전체 컬럼 수      : {df.shape[1]}')
print(f'공정 변수 수      : {len(process_cols)}')
print(f'라만 분광 채널 수  : {len(raman_cols)}')
print(f'라만 웨이브넘버    : {min(int(c) for c in raman_cols)} ~ {max(int(c) for c in raman_cols)} cm\u207b\u00b9')
print()
print('공정 변수 목록:')
for i, c in enumerate(process_cols):
    print(f'  [{i:2d}] {c}')

In [ ]:
batch_lengths = df[BATCH_COL].value_counts().sort_index()

print(f'배치 수          : {batch_lengths.shape[0]}')
print(f'타임스텝 수 (min) : {batch_lengths.min()}')
print(f'타임스텝 수 (max) : {batch_lengths.max()}')
print(f'타임스텝 수 (mean): {batch_lengths.mean():.1f}')

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(batch_lengths.index, batch_lengths.values, color='steelblue', width=0.8)
ax.set_xlabel('Batch ID')
ax.set_ylabel('Timesteps')
ax.set_title('Timesteps per Batch')
plt.tight_layout()
plt.show()

## 3. 배치 통계 분석

In [ ]:
fault_counts = df_stats[FAULT_COL].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].pie(
    fault_counts.values,
    labels=['No-Fault', 'Fault'],
    autopct='%1.0f%%',
    colors=['#4C9BE8', '#E8644C'],
    startangle=90
)
axes[0].set_title('Fault Batch Ratio')

for label, color in zip([0, 1], ['#4C9BE8', '#E8644C']):
    subset = df_stats[df_stats[FAULT_COL] == label][YIELD_COL]
    axes[1].hist(subset, bins=15, alpha=0.7, color=color,
                 label='No-Fault' if label == 0 else 'Fault')
axes[1].set_xlabel('Penicillin Yield Total (kg)')
axes[1].set_ylabel('Count')
axes[1].set_title('Yield Distribution (Fault vs No-Fault)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(df_stats.groupby(FAULT_COL)[YIELD_COL].describe().round(0))

## 4. 공정 변수 시계열 탐색

In [ ]:
key_vars = [
    'pH(pH:pH)',
    'Dissolved oxygen concentration(DO2:mg/L)',
    'Temperature(T:K)',
    'Penicillin concentration(P:g/L)',
    'Substrate concentration(S:g/L)',
    'Vessel Volume(V:L)',
]

fault_batch_ids    = df_stats[df_stats[FAULT_COL] == 1][BATCH_COL_STATS].values[:1]
no_fault_batch_ids = df_stats[df_stats[FAULT_COL] == 0][BATCH_COL_STATS].values[:2]

fig, axes = plt.subplots(len(key_vars), 1, figsize=(13, 14), sharex=False)

for ax, var in zip(axes, key_vars):
    for bid in no_fault_batch_ids:
        sub = df[df[BATCH_COL] == bid]
        ax.plot(sub[TIME_COL], sub[var], color='#4C9BE8', alpha=0.8, linewidth=1.2, label='No-Fault')
    for bid in fault_batch_ids:
        sub = df[df[BATCH_COL] == bid]
        ax.plot(sub[TIME_COL], sub[var], color='#E8644C', alpha=0.9, linewidth=1.4, linestyle='--', label='Fault')
    ax.set_ylabel(var.split('(')[0].strip(), fontsize=9)
    ax.tick_params(axis='both', labelsize=8)

axes[0].legend(loc='upper right', fontsize=9)
axes[-1].set_xlabel('Time (h)')
fig.suptitle('Key Process Variables Time-Series (Sample Batches)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
var = 'Penicillin concentration(P:g/L)'
grouped  = df.groupby(TIME_COL)[var]
mean_val = grouped.mean()
std_val  = grouped.std()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mean_val.index, mean_val.values, color='steelblue', linewidth=1.5, label='Mean')
ax.fill_between(mean_val.index, mean_val - std_val, mean_val + std_val,
                alpha=0.25, color='steelblue', label='\u00b11 Std')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Penicillin (g/L)')
ax.set_title('Penicillin Concentration \u2014 All Batches Mean \u00b1 Std')
ax.legend()
plt.tight_layout()
plt.show()

## 5. 라만 분광 데이터 탐색

In [ ]:
# PAT_ref: 0 = 라만 없음, 1 = 라만 기록됨
print('PAT_ref 분포:')
print(df[PAT_COL].value_counts())
print(f'라만 기록된 행 비율: {(df[PAT_COL] == 1).mean()*100:.1f}%')

In [ ]:
df_raman    = df[df[PAT_COL] == 1].copy()
raman_data  = df_raman[raman_cols].astype(float)
wavenumbers = np.array([int(c) for c in raman_cols])

print(f'라만 데이터 shape: {raman_data.shape}')

mean_spectrum = raman_data.mean(axis=0).values
std_spectrum  = raman_data.std(axis=0).values

all_fault_ids    = df_stats[df_stats[FAULT_COL] == 1][BATCH_COL_STATS].values
all_no_fault_ids = df_stats[df_stats[FAULT_COL] == 0][BATCH_COL_STATS].values

raman_fault    = df_raman[df_raman[BATCH_COL].isin(all_fault_ids)][raman_cols].astype(float).mean()
raman_no_fault = df_raman[df_raman[BATCH_COL].isin(all_no_fault_ids)][raman_cols].astype(float).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 7))

axes[0].plot(wavenumbers, mean_spectrum, color='steelblue', linewidth=0.9)
axes[0].fill_between(wavenumbers, mean_spectrum - std_spectrum, mean_spectrum + std_spectrum,
                     alpha=0.25, color='steelblue')
axes[0].set_xlabel('Wavenumber (cm\u207b\u00b9)')
axes[0].set_ylabel('Intensity')
axes[0].set_title('Mean Raman Spectrum \u00b1 1 Std')

axes[1].plot(wavenumbers, raman_no_fault.values, color='#4C9BE8', linewidth=0.9, label='No-Fault')
axes[1].plot(wavenumbers, raman_fault.values,    color='#E8644C', linewidth=0.9, label='Fault', linestyle='--')
axes[1].set_xlabel('Wavenumber (cm\u207b\u00b9)')
axes[1].set_ylabel('Intensity')
axes[1].set_title('Raman Spectrum \u2014 Fault vs No-Fault Mean Comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
raman_var = raman_data.var(axis=0)
raman_var.index = wavenumbers
top20 = raman_var.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(top20.index.astype(str), top20.values, color='steelblue')
ax.set_xlabel('Wavenumber (cm\u207b\u00b9)')
ax.set_ylabel('Variance')
ax.set_title('Raman Spectroscopy \u2014 Top 20 High-Variance Channels')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. 결측치 분석

In [ ]:
missing = df[process_cols].isnull().mean().sort_values(ascending=False)
missing_nonzero = missing[missing > 0]

print(f'결측치 있는 공정 변수 수: {len(missing_nonzero)} / {len(process_cols)}')
print()
print(missing_nonzero.to_string())

if len(missing_nonzero) > 0:
    fig, ax = plt.subplots(figsize=(10, 3))
    missing_nonzero.plot(kind='bar', ax=ax, color='#E8644C')
    ax.set_ylabel('Missing Ratio')
    ax.set_title('Missing Value Ratio in Process Variables')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# 오프라인 변수는 간헐적으로 측정됨
offline_vars = [c for c in process_cols if 'offline' in c.lower() or 'Offline' in c]
print('Offline 변수:', offline_vars)

batch1 = df[df[BATCH_COL] == 1]
for v in offline_vars[:3]:
    measured = batch1[v].notna().sum()
    print(f'  {v.split("(")[0].strip()}: {measured}/{len(batch1)} 타임스텝에서 측정')

## 7. Fault vs Normal 배치 비교

In [ ]:
all_fault_ids    = df_stats[df_stats[FAULT_COL] == 1][BATCH_COL_STATS].values
all_no_fault_ids = df_stats[df_stats[FAULT_COL] == 0][BATCH_COL_STATS].values

print('Fault 배치 ID:', sorted(all_fault_ids))

df_fault    = df[df[BATCH_COL].isin(all_fault_ids)]
df_no_fault = df[df[BATCH_COL].isin(all_no_fault_ids)]

compare_vars = [
    'pH(pH:pH)',
    'Dissolved oxygen concentration(DO2:mg/L)',
    'Temperature(T:K)',
    'Penicillin concentration(P:g/L)',
]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, var in zip(axes, compare_vars):
    for group, label, color in [
        (df_no_fault, 'No-Fault', '#4C9BE8'),
        (df_fault,    'Fault',    '#E8644C')
    ]:
        mean_t = group.groupby(TIME_COL)[var].mean()
        std_t  = group.groupby(TIME_COL)[var].std()
        ax.plot(mean_t.index, mean_t.values, color=color, linewidth=1.2, label=label)
        ax.fill_between(mean_t.index, mean_t - std_t, mean_t + std_t, alpha=0.15, color=color)
    ax.set_title(var.split('(')[0].strip(), fontsize=9)
    ax.set_xlabel('Time (h)', fontsize=8)
    ax.tick_params(labelsize=8)

axes[0].legend(fontsize=9)
fig.suptitle('Fault vs No-Fault \u2014 Mean Time-Series Comparison', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
numeric_vars = [
    'pH(pH:pH)',
    'Dissolved oxygen concentration(DO2:mg/L)',
    'Temperature(T:K)',
    'Penicillin concentration(P:g/L)',
    'Substrate concentration(S:g/L)',
    'Vessel Volume(V:L)',
    'Aeration rate(Fg:L/h)',
    'Agitator RPM(RPM:RPM)',
    'carbon dioxide percent in off-gas(CO2outgas:%)',
    'Oxygen Uptake Rate(OUR:(g min^{-1}))',
    'Generated heat(Q:kJ)',
]

corr = df_no_fault[numeric_vars].corr()
short_names = [c.split('(')[0].strip() for c in numeric_vars]
corr.index   = short_names
corr.columns = short_names

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Process Variable Correlation (No-Fault Batches)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 8. 요약 및 연구 방향 메모

In [ ]:
print('=' * 55)
print('EDA 요약')
print('=' * 55)
print(f'  배치 수           : 100 (Fault: 10개, Normal: 90개)')
print(f'  전체 행 수        : {len(df):,}')
print(f'  공정 변수 수       : {len(process_cols)}')
print(f'  라만 분광 채널 수   : {len(raman_cols)} (201~2400 cm\u207b\u00b9)')
print(f'  라만 기록 비율     : {(df[PAT_COL]==1).mean()*100:.1f}%')
print(f'  수율 평균          : {df_stats[YIELD_COL].mean():,.0f} kg')
print(f'  수율 표준편차      : {df_stats[YIELD_COL].std():,.0f} kg')
print('=' * 55)

### 관찰 사항 (EDA 실행 후 직접 기록)

- [ ] Fault 배치와 Normal 배치 간 어떤 변수에서 가장 큰 차이가 관찰되었는가?
- [ ] 라만 분광에서 분산이 큰 채널은 어떤 물질과 연관될 수 있는가?
- [ ] 오프라인 측정 변수의 결측 패턴이 연구에 어떤 영향을 주는가?
- [ ] 배치 간 타임스텝 수 차이가 있는가? 있다면 정렬(alignment) 전략이 필요한가?

### 잠재적 연구 주제

- **주제 A**: 라만 분광 기반 페니실린 농도 소프트 센서 (Soft Sensor)
- **주제 B**: 딥러닝 기반 이상 감지 (Anomaly Detection) — LSTM-AE, Transformer-AE
- **주제 C**: 멀티모달 융합 (공정 변수 + 라만) 기반 수율 예측
- **주제 D**: 배치 공정 모니터링 — MSPC (PCA 기반) vs 딥러닝 비교

> 연구 주제는 EDA 관찰 결과를 바탕으로 확정할 것.